# 03b — Global XGBoost regressor (Farm A)

**Fleet-level supervised** model on **`wind-farm-a-features`**, default target **`hours_to_next_operator_warning`** (soonest of next anomaly window vs anomaly-linked downtime; see 02 CELL 8). **Leave-one-event-out (LOOCV)** over **`USABLE_ANOMALY_EVENT_IDS`** (must match **`02_feature_engineering` CELL 1** — Farm A: **all 12** anomaly `event_id`s **0, 10, 22, 26, 40, 42, 45, 51, 68, 72, 73, 84**).

This path is **separate** from **03a** (per-turbine Isolation Forest on `if_fit_eligible`). The Spark filter keeps **`status_type_id` ∈ `SUPERVISED_NORMAL_STATUS`** (default `[0]`, aligned with **02** Farm A `normal_status`) so horizon regression trains on **normal operation** rows only — even though **02** CELL 4 retains all statuses for feature engineering. **`is_usable_supervised_next`** selects rows whose **next** anomaly is in the usable set.

**Imbalance** — Many rows have large horizons vs few near-event rows; consider **sample weights** or **weighted metrics** (see **NOTEBOOKS.md**); **`scale_pos_weight`** is for classification, not this regressor.

**Separate targets for analysis** — The feature table also carries **`hours_to_next_anomaly`** (logged window) and **`hours_to_anomaly_linked_downtime`** (failure transition). Set **`TARGET`** to either column in CELL 1 to compare which horizon is more predictable; keep **`hours_to_next_operator_warning`** as the unified operational objective.

**Interpretation (LOOCV)** — With **12** folds, each holdout tests one failure context while training on the other eleven; **report every fold’s RMSE** (and spread), not only a single aggregate. Generalization to **new failure types** outside this fleet remains **not** demonstrated. Fault families can still show **heterogeneous** pre-failure sensor patterns; a single global model is an honest **scaffold**. Optional future work: Farms B/C, failure-type features, component-specific trends.

**MLflow** — One parent run aggregates fold metrics; nested runs per holdout event.

**Stage 4 outputs (CELL 4)** — After LOOCV: **final** `XGBRegressor` trained on **all** supervised rows (all 12 events); **SHAP** global importance table **`wind-farm-a-xgb-shap-importance`** (`feature`, `mean_abs_shap`); **scores Delta** **`wind-farm-a-xgb-scores`** with **`xgb_pred_hours_to_operator_warning`**, **`xgb_risk_score_tiered`** (same tier semantics as 02), **`xgb_risk_score_norm`** (min–max inverted on the scored batch); MLflow run **`XGB_Farm_A_final_production`** logs **`model_final_all_events`** + SHAP CSV artifact. Restart kernel after pip if needed.


In [ ]:
%pip install mlflow xgboost shap


In [ ]:
dbutils.library.restartPython()


In [ ]:
# CELL 1 — Imports and config (align USABLE_* + TRAIN_END with 02_feature_engineering CELL 1)

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import warnings

warnings.filterwarnings("ignore")

CATALOG = "wind-turbine-silver"
FEATURE_TABLE = "wind-farm-a-features"

USABLE_ANOMALY_EVENT_IDS = frozenset({0, 10, 22, 26, 40, 42, 45, 51, 68, 72, 73, 84})
TARGET = "hours_to_next_operator_warning"
# Must match 02 FARM_CONFIG["a"]["normal_status"] — horizon regression uses normal rows only
SUPERVISED_NORMAL_STATUS = [0]

OUTPUT_TABLE_SCORES = "wind-farm-a-xgb-scores"
OUTPUT_TABLE_SHAP = "wind-farm-a-xgb-shap-importance"
# Same hyperparams as LOOCV folds — final model uses all training rows
XGB_PARAMS = dict(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
SHAP_N_BACKGROUND = 200
SHAP_N_EXPLAIN = 3000
PRED_CHUNK_ROWS = 100_000

mlflow.set_experiment(
    "/Users/"
    + spark.sql("SELECT current_user()").first()[0]
    + "/DSMLC-Final-Comp-2026-xgboost"
)
mlflow.sklearn.autolog(disable=True)

EXCLUDE_COLS = {
    "asset_id",
    "time_stamp",
    "id",
    "train_test",
    "status_type_id",
    "risk_score",
    "hours_to_next_anomaly",
    "hours_to_anomaly_linked_downtime",
    "risk_score_anomaly_linked_downtime",
    "hours_to_next_operator_warning",
    "risk_score_operator_warning",
    "next_anomaly_event_id",
    "hours_to_failure_onset",
    "risk_score_onset",
    "next_failure_onset_ts",
    "next_onset_event_overlaps",
    "in_overlap_event_period",
    "is_usable_supervised_next",
    "if_fit_eligible",
    "in_anomaly_window",
}


In [ ]:
# CELL 2 — Load features and build feature matrix columns

df_spark = spark.table(f"`{CATALOG}`.`{FEATURE_TABLE}`")

required = [
    "asset_id",
    "time_stamp",
    "id",
    TARGET,
    "next_anomaly_event_id",
    "is_usable_supervised_next",
    "status_type_id",
]
missing = [c for c in required if c not in df_spark.columns]
if missing:
    raise ValueError(
        f"Missing columns {missing} — run 02_feature_engineering through CELL 8"
    )

feature_cols = [c for c in df_spark.columns if c not in EXCLUDE_COLS]
print(f"Feature columns: {len(feature_cols)}")

base = df_spark.filter(
    F.col(TARGET).isNotNull()
    & F.col("is_usable_supervised_next")
    & F.col("next_anomaly_event_id").isin(*sorted(USABLE_ANOMALY_EVENT_IDS))
    & F.col("status_type_id").isin(SUPERVISED_NORMAL_STATUS)
)
print(f"Usable supervised rows (Spark): {base.count():,}")


In [ ]:
# CELL 3 — Leave-one-event-out CV + global XGBoost (materialize with care on large data)

pdf = base.select(*(feature_cols + [TARGET, "next_anomaly_event_id"])).toPandas()
pdf["next_anomaly_event_id"] = pdf["next_anomaly_event_id"].astype("int64")

X = pdf[feature_cols].to_numpy(dtype=float)
y = pdf[TARGET].to_numpy(dtype=float)
groups = pdf["next_anomaly_event_id"].to_numpy()

fold_rmses = []
with mlflow.start_run(run_name="XGB_LOOCV_Farm_A_parent"):
    mlflow.log_param("usable_event_ids", str(sorted(USABLE_ANOMALY_EVENT_IDS)))
    mlflow.log_param("n_folds_planned", len(USABLE_ANOMALY_EVENT_IDS))

    for fold_idx, holdout in enumerate(sorted(USABLE_ANOMALY_EVENT_IDS)):
        train_mask = groups != holdout
        test_mask = groups == holdout
        if train_mask.sum() == 0 or test_mask.sum() == 0:
            print(f"Skip fold {holdout}: insufficient rows")
            continue

        X_tr, y_tr = X[train_mask], y[train_mask]
        X_te, y_te = X[test_mask], y[test_mask]

        model = XGBRegressor(**XGB_PARAMS)

        with mlflow.start_run(run_name=f"loocv_holdout_event_{holdout}", nested=True):
            mlflow.log_param("loocv_fold_index", fold_idx)
            mlflow.log_param("holdout_event_id", holdout)
            mlflow.log_param("n_train", int(train_mask.sum()))
            mlflow.log_param("n_test", int(test_mask.sum()))
            model.fit(X_tr, y_tr)
            pred = model.predict(X_te)
            rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
            mlflow.log_metric("rmse_test", rmse)
            mlflow.sklearn.log_model(model, artifact_path="model")
            fold_rmses.append(rmse)
            print(f"fold={fold_idx} holdout={holdout} RMSE={rmse:.4f}")

    if fold_rmses:
        mlflow.log_metric("rmse_loocv_mean", float(np.mean(fold_rmses)))
        mlflow.log_metric("rmse_loocv_std", float(np.std(fold_rmses)))
        mlflow.log_metric("rmse_loocv_n_folds", float(len(fold_rmses)))
        print(
            f"LOOCV summary: mean RMSE={np.mean(fold_rmses):.4f}, std={np.std(fold_rmses):.4f}, n={len(fold_rmses)}"
        )

print("LOOCV complete.")


In [ ]:
# CELL 4 — Final model (all usable events), SHAP global importance, Delta outputs (Stage 4)

import shap


def risk_tier_from_pred_hours(h: np.ndarray) -> np.ndarray:
    """Matches 02 RISK_TIER_HOURS semantics: lower hours => higher risk in {0,0.3,0.6,0.9,1.0}."""
    h = np.asarray(h, dtype=float)
    h = np.where(np.isnan(h), np.inf, h)
    return np.select(
        [h <= 24, h <= 72, h <= 168, h <= 336, h > 336],
        [1.0, 0.9, 0.6, 0.3, 0.0],
        default=0.0,
    )


def minmax_risk_inverted(h: np.ndarray) -> np.ndarray:
    """0–1 score: 1 = most urgent (smallest positive hours)."""
    h = np.asarray(h, dtype=float)
    finite = h[np.isfinite(h) & (h >= 0)]
    if finite.size == 0:
        return np.zeros_like(h, dtype=float)
    lo, hi = float(np.min(finite)), float(np.max(finite))
    if hi <= lo:
        return np.ones_like(h, dtype=float) * 0.5
    z = (np.clip(h, lo, hi) - lo) / (hi - lo)
    return 1.0 - z


# --- 1) Train on all supervised normal rows (12 events) ---
model_final = XGBRegressor(**XGB_PARAMS)
model_final.fit(X, y)
print(f"Final model: trained on n={len(y):,} rows (all usable events).")

# --- 2) SHAP (global mean |SHAP| on a random subset) ---
rng = np.random.default_rng(42)
n_bg = min(SHAP_N_BACKGROUND, len(X))
bg_idx = rng.choice(len(X), size=n_bg, replace=False)
X_bg = X[bg_idx]
explainer = shap.TreeExplainer(model_final, data=X_bg)
n_ex = min(SHAP_N_EXPLAIN, len(X))
ex_idx = rng.choice(len(X), size=n_ex, replace=False)
X_ex = X[ex_idx]
sv = explainer(X_ex)
vals = np.asarray(getattr(sv, "values", sv))
if vals.ndim == 3:
    mean_abs = np.abs(vals).mean(axis=(0, 2))
else:
    mean_abs = np.abs(vals).mean(axis=0)
shap_pdf = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs})
shap_pdf = shap_pdf.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

shap_csv = "/tmp/xgb_shap_importance.csv"
shap_pdf.to_csv(shap_csv, index=False)

spark.createDataFrame(shap_pdf).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"`{CATALOG}`.`{OUTPUT_TABLE_SHAP}`")
print(f"Saved SHAP importance -> `{CATALOG}`.`{OUTPUT_TABLE_SHAP}`")

# --- 3) Score full feature table (all rows, all statuses) for deployment joins ---
meta_cols = ["asset_id", "time_stamp", "id", "train_test", "status_type_id"]
for c in ("next_anomaly_event_id", "if_fit_eligible", "in_anomaly_window"):
    if c in df_spark.columns:
        meta_cols.append(c)

score_spark = df_spark.select(*(meta_cols + feature_cols)).fillna(0, subset=feature_cols)
score_pdf = score_spark.toPandas()
pred_all = np.empty(len(score_pdf), dtype=float)
for start in range(0, len(score_pdf), PRED_CHUNK_ROWS):
    sl = slice(start, start + PRED_CHUNK_ROWS)
    pred_all[sl] = model_final.predict(
        score_pdf.loc[sl, feature_cols].to_numpy(dtype=float)
    )

score_pdf["xgb_pred_hours_to_operator_warning"] = pred_all
score_pdf["xgb_risk_score_tiered"] = risk_tier_from_pred_hours(pred_all)
score_pdf["xgb_risk_score_norm"] = minmax_risk_inverted(pred_all)

out_cols = meta_cols + [
    "xgb_pred_hours_to_operator_warning",
    "xgb_risk_score_tiered",
    "xgb_risk_score_norm",
]
out_pdf = score_pdf[[c for c in out_cols if c in score_pdf.columns]]

spark.createDataFrame(out_pdf).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"`{CATALOG}`.`{OUTPUT_TABLE_SCORES}`")
print(f"Saved scores -> `{CATALOG}`.`{OUTPUT_TABLE_SCORES}` (rows={len(out_pdf):,})")

# --- 4) MLflow: final model + SHAP artifact ---
with mlflow.start_run(run_name="XGB_Farm_A_final_production"):
    mlflow.log_param("n_train_supervised", len(y))
    mlflow.log_param("n_score_rows", len(out_pdf))
    mlflow.log_param("output_scores_table", OUTPUT_TABLE_SCORES)
    mlflow.log_param("output_shap_table", OUTPUT_TABLE_SHAP)
    mlflow.sklearn.log_model(model_final, artifact_path="model_final_all_events")
    mlflow.log_artifact(shap_csv)

print("CELL 4 complete: final model, SHAP table, risk scores Delta, MLflow run.")
